In [82]:
import pandas as pd
import re

In [83]:
pd.set_option('display.max_colwidth', None)

In [84]:
sin_nombre = pd.read_csv('../data/raw/sin_nombre_all.csv')

In [85]:
sin_nombre.shape

(397, 28)

In [86]:
trash_words = 'VACCINE PLATFORM|patent|regulatory elements|METHODS|Modified Microbial Nucleic Acid'

In [87]:
trash_mask = sin_nombre['GenBank_Title'].str.contains(trash_words, case=False, na=False, regex=True)

In [88]:
trash_rows = sin_nombre[trash_mask]

In [89]:
print(f'Broj pronađenih redova: {len(trash_rows)}')

Broj pronađenih redova: 31


In [90]:
display(trash_rows[['Accession', 'GenBank_Title','Segment']].head(10))

,Accession,GenBank_Title,Segment
15,PG340915.1,JP 2023515355-A/7: RAPID VACCINE PLATFORM,NaN
16,PG340916.1,JP 2023515355-A/8: RAPID VACCINE PLATFORM,NaN
17,PG340917.1,JP 2023515355-A/9: RAPID VACCINE PLATFORM,NaN
18,PG340918.1,JP 2023515355-A/10: RAPID VACCINE PLATFORM,NaN
19,PL092163.1,"KR 1020240003760-A/15: New regulatory elements for enhancing RNA stability or mRNA translation, ZCCHC2 interacting with the same, and use thereof",NaN
20,PL092227.1,"KR 1020240003760-A/79: New regulatory elements for enhancing RNA stability or mRNA translation, ZCCHC2 interacting with the same, and use thereof",NaN
21,PL092228.1,"KR 1020240003760-A/80: New regulatory elements for enhancing RNA stability or mRNA translation, ZCCHC2 interacting with the same, and use thereof",NaN
22,PL092350.1,"KR 1020240003761-A/15: A method of screening regulatory elements for enhancing mRNA translation, new regulatory elements according to the method, and use thereof",NaN
23,PL092414.1,"KR 1020240003761-A/79: A method of screening regulatory elements for enhancing mRNA translation, new regulatory elements according to the method, and use thereof",NaN
24,PL092415.1,"KR 1020240003761-A/80: A method of screening regulatory elements for enhancing mRNA translation, new regulatory elements according to the method, and use thereof",NaN


Uklanjanje patenata, dijagnostičkih metoda, vakcinacionih platformi i veštačkih modifikovanih sekvenci (KR/JP patenti, regulatorni elementi iRNK) kako bi se eliminisao šum i zadržali isključivo prirodni S, M i L segmenti hantavirusa.

In [91]:
sin_nombre = sin_nombre[
    ~sin_nombre['GenBank_Title'].str.contains(trash_words, case=False, na=False)].copy()

In [92]:
sin_nombre.shape

(366, 28)

In [93]:
sin_nombre['Segment'].value_counts(dropna=False)

Segment
NaN    138
M      124
S       63
L       41
Name: count, dtype: int64

In [94]:
display(sin_nombre[sin_nombre['Segment'].isna()][['Accession', 'GenBank_Title']].head(50))

,Accession,GenBank_Title
194,KX066110.1,"Orthohantavirus sinnombreense strain HV B0530012 nucleocapsid protein (N) gene, partial cds"
195,KX066111.1,"Orthohantavirus sinnombreense strain HV B0770007 nucleocapsid protein (N) gene, partial cds"
196,KX066112.1,"Orthohantavirus sinnombreense strain HV B0530013 nucleocapsid protein (N) gene, partial cds"
197,KX066113.1,"Orthohantavirus sinnombreense strain HV B0530015 nucleocapsid protein (N) gene, partial cds"
198,KX066114.1,"Orthohantavirus sinnombreense strain HV B0770009 nucleocapsid protein (N) gene, partial cds"
199,KX066117.1,"Orthohantavirus sinnombreense strain HV B0770021 nucleocapsid protein (N) gene, partial cds"
200,KX066118.1,"Orthohantavirus sinnombreense strain HV B0770014 nucleocapsid protein (N) gene, partial cds"
201,KX066119.1,"Orthohantavirus sinnombreense strain HV B0530001 nucleocapsid protein (N) gene, partial cds"
202,KX066120.1,"Orthohantavirus sinnombreense strain HV B0770015 nucleocapsid protein (N) gene, partial cds"
203,KX066121.1,"Orthohantavirus sinnombreense strain HV B0770016 nucleocapsid protein (N) gene, partial cds"


In [95]:
is_nan_segment = sin_nombre['Segment'].isna()

In [96]:
sin_nombre.loc[
    is_nan_segment & sin_nombre['GenBank_Title'].str.contains('nucleocapsid|S RNA segment|S segment|nucleocapsin', case=False, na=False),
    'Segment'
] = 'S'

sin_nombre.loc[
    is_nan_segment & sin_nombre['GenBank_Title'].str.contains('glycoprotein|G1 protein gene|G2 protein gene|M segment', case=False, na=False),
    'Segment'
] = 'M'

sin_nombre.loc[
    is_nan_segment & sin_nombre['GenBank_Title'].str.contains('L segment', case=False, na=False),
    'Segment'
] = 'L'

In [97]:
sin_nombre['Segment'].value_counts(dropna=False)

Segment
M    214
S    108
L     44
Name: count, dtype: int64

In [98]:
cols = ['Accession', 'Segment',  'Nuc_Completeness', 'Species']

In [99]:
all_sequences = sin_nombre[cols]
all_sequences.to_csv('../data/processed_sequences/all_sequences/sin_nombre_all.csv')

In [100]:
complete_sequences = all_sequences[all_sequences['Nuc_Completeness'] == 'complete'].copy()
complete_sequences.to_csv('../data/processed_sequences/complete_sequences/sin_nombre_complete.csv')